# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [21]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # START (r, c)
        self.start = (4, 0)

        # Walls (Estanterías)
        self.walls = {
            (0, 1), (0, 2), (0, 3), (0, 4),
            (1, 1), (1, 4),
            (2, 1), (2, 4),
            (3, 1), (3, 4),
            (4, 1), (4, 2), (4, 3)
        }

        # Slippery states
        self.slippery_states = {
            (1, 0), (1, 2), (1, 3),
            (2, 0), (2, 2), (2, 3),
            (3, 0), (3, 2), (3, 3)
        }

        # Terminal states: (row, col): reward
        self.terminal_states = {
            (0, 5): 10.0,  # Entrega
            (4, 5): 2.0,   # Carga
            (3, 5): -10.0  # Peligro mortal
        }

        # Danger states (non-terminal with -3 reward)
        self.danger_states = {
            (0, 0),
            (2, 5)
        }

        self.living_reward = -1.0 # Costo por paso
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        return 0 <= r < self.height and \
               0 <= c < self.width and \
               state not in self.walls

    def states(self):
        S = set()
        for r in range(self.height):
            for c in range(self.width):
                state = (r, c)
                if self.is_valid_state(state):
                    S.add(state)
        return sorted(list(S))

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        elif state in self.danger_states:
            return -3.0
        else:
            return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [] # No transitions from terminal states, as per standard MDP formulation

        transitions = {}
        r, c = state
        dr, dc = action

        # Define probabilities based on slipperiness
        if state in self.slippery_states:
            p_straight = 0.60
            p_deviation = 0.20
        else:
            p_straight = 0.90
            p_deviation = 0.05

        # Calculate actual next states for desired action and deviations
        possible_actions = [
            ((dr, dc), p_straight),         # Desired action
            ((-dc, dr), p_deviation),       # Left deviation
            ((dc, -dr), p_deviation)        # Right deviation
        ]

        for (adr, adc), prob in possible_actions:
            next_r, next_c = r + adr, c + adc
            next_state_candidate = (next_r, next_c)

            if self.is_valid_state(next_state_candidate):
                actual_next_state = next_state_candidate
            else:
                actual_next_state = state # Stays in current state if hits wall or boundary

            # Aggregate probabilities if multiple actions lead to the same actual_next_state
            transitions[actual_next_state] = transitions.get(actual_next_state, 0.0) + prob

        return [(s, p) for s, p in transitions.items()]


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [22]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    if grid.is_terminal(s): # Skip terminal states as they have no outgoing transitions
        continue
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12, f"Sum of probabilities for state {s}, action {a} is {total}"

print("✓ Todas las distribuciones de transición suman 1 (ignorando estados terminales).")

Número de estados: 17
✓ Todas las distribuciones de transición suman 1 (ignorando estados terminales).



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [34]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    sum_of_probs_times_values = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        sum_of_probs_times_values += prob * V.get(next_state, 0.0) # V.get handles terminal states if they are not in V
    return sum_of_probs_times_values


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states() if not grid.is_terminal(s)} # Initialize V(s) = 0 for non-terminal states

    # For terminal states, their value is their reward (they don't transition)
    for s in grid.states():
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    n_iter = 0
    for _ in range(max_iter):
        delta = 0
        n_iter += 1
        for s in grid.states():
            if grid.is_terminal(s):
                continue # Terminal states have fixed values

            v_old = V[s]

            # Calculate R(s) + gamma * max_a (sum_{s'} T(s,a,s') V(s'))
            # find max_a for this state
            max_q_value = -float('inf')
            for action in grid.actions:
                q_value = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
                if q_value > max_q_value:
                    max_q_value = q_value

            V[s] = max_q_value
            delta = max(delta, abs(v_old - V[s]))

        if delta < threshold:
            break

    return V, n_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    epsilon_tie_break = 0.1 # A more permissive epsilon for comparing Q-values for ties

    for s in grid.states():
        if grid.is_terminal(s):
            continue # No policy for terminal states

        best_action = None
        max_q_value = -float('inf')
        best_action_idx = -1 # To store the index for consistent tie-breaking

        for idx, action in enumerate(grid.actions):
            q_value = expected_next_value(grid, s, action, V)

            # If current q_value is strictly better, OR
            # if it's approximately equal and this action has a lower index (for tie-breaking)
            if (q_value > max_q_value + epsilon_tie_break) or \
               (best_action is None) or \
               (abs(q_value - max_q_value) < epsilon_tie_break and idx < best_action_idx):

                max_q_value = q_value
                best_action = action
                best_action_idx = idx

        policy[s] = best_action
    return policy


## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [35]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states() if not grid.is_terminal(s)}

    # For terminal states, their value is their reward
    for s in grid.states():
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)

    n_iter = 0
    for _ in range(max_iter):
        delta = 0
        n_iter += 1
        for s in grid.states():
            if grid.is_terminal(s):
                continue

            v_old = V[s]
            action = policy.get(s) # Get action from current policy

            if action is None: # If a state is not in policy (e.g., unreachable yet in first iteration), skip or assign a default
                V[s] = grid.get_reward(s) # Or some other handling
                delta = max(delta, abs(v_old - V[s]))
                continue

            # Calculate V_k+1(s) = R(s) + gamma * sum_{s'} T(s,pi(s),s') V_k(s')
            expected_val = expected_next_value(grid, s, action, V)
            V[s] = grid.get_reward(s) + grid.gamma * expected_val
            delta = max(delta, abs(v_old - V[s]))

        if delta < threshold:
            break

    return V, n_iter


def policy_improvement(grid, V):
    new_policy = {}
    epsilon_tie_break = 0.1 # A more permissive epsilon for comparing Q-values for ties

    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        max_q_value = -float('inf')
        best_action_idx = -1 # To store the index for consistent tie-breaking

        for idx, action in enumerate(grid.actions):
            q_value = expected_next_value(grid, s, action, V)

            # If current q_value is strictly better, OR
            # if it's approximately equal and this action has a lower index (for tie-breaking)
            if (q_value > max_q_value + epsilon_tie_break) or \
               (best_action is None) or \
               (abs(q_value - max_q_value) < epsilon_tie_break and idx < best_action_idx):

                max_q_value = q_value
                best_action = action
                best_action_idx = idx

        new_policy[s] = best_action
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria (e.g., all UP)
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    # Initialize with default action even for terminal states, will be overwritten.
    # Terminal states won't be in the policy anyway, but for the sake of completeness
    # or if `grid.states()` includes terminal states, then filter them.
    for s in grid.states():
        if grid.is_terminal(s):
            if s in policy: del policy[s] # Ensure terminal states are not in policy


    history = []
    n_iter = 0

    for _ in range(max_iter):
        n_iter += 1
        # 2. evaluación
        V, eval_iter = policy_evaluation(grid, policy, threshold=threshold)
        history.append(f"Eval Iter: {eval_iter}")

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        # 4. repetir hasta estabilidad
        # Check if policy has converged
        policy_stable = True
        for s in grid.states():
            if not grid.is_terminal(s):
                # Compare policies considering None for uninitialized states (though shouldn't happen after first PI step)
                if policy.get(s) != new_policy.get(s):
                    policy_stable = False
                    break

        if policy_stable:
            break

        policy = new_policy
        history.append("Policy changed")

    return policy, V, history


## Parte 4 — Visualización y comparación


In [26]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [37]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 89

Valores:
-12.197 |   WALL   |   WALL   |   WALL   |   WALL   | +10.000
 -9.999 |   WALL   | -10.000 | -10.000 |   WALL   |  +7.802
 -9.999 |   WALL   | -10.000 | -10.000 |   WALL   |  +3.648
 -9.999 |   WALL   | -10.000 | -10.000 |   WALL   | -10.000
 -9.999 |   WALL   |   WALL   |   WALL   |  +0.681 |  +2.000

Política:
 ↓  |  #  |  #  |  #  |  #  | +10
 ↓  |  #  |  ↑  |  ↑  |  #  |  ↑ 
 ↑  |  #  |  ↑  |  ↑  |  #  |  ↑ 
 ↑  |  #  |  ↑  |  ↑  |  #  | -10
 ↑  |  #  |  #  |  #  |  →  | +2

=== POLICY ITERATION ===
Historia: ['Eval Iter: 99', 'Policy changed', 'Eval Iter: 89', 'Policy changed', 'Eval Iter: 81']

Valores:
-12.198 |   WALL   |   WALL   |   WALL   |   WALL   | +10.000
-10.000 |   WALL   |  -9.999 |  -9.999 |   WALL   |  +7.802
-10.000 |   WALL   |  -9.999 |  -9.999 |   WALL   |  +3.648
-10.000 |   WALL   |  -9.999 |  -9.999 |   WALL   | -10.000
-10.000 |   WALL   |   WALL   |   WALL   |  +0.681 |  +2.000

Política:
 ↓  |  #  |  #  |  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?

La entrega +10 tiene valores de estado más altos, por lo que es la recompensa más deseable.

2. ¿Por qué una recompensa menor podría ser óptima?

Puede ser óptima si es mucho más fácil de alcanzar (ruta más corta).

3. ¿En qué estados el piso resbaloso cambia la decisión?

En estados donde una desviación accidental podría llevar a un peligro mayor. El agente tiende a tomar acciones más conservadoras para mitigar el posible riesgo.

4. ¿Qué papel cumple el costo por paso `-1`?

fuerza al robot a encontrar la ruta más corta y eficiente posible para alcanzar recompensas positivas y evitar el que deambule sin rumbo.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

Ya que las probabilidades varian segun el estado del entorno.

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
